In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        if requirement is None:  # a GPU-only package on a CPU runtime
            continue
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, cpu_requirement, _, match) in pinned.items():
    if not gpu and cpu_requirement is None:
        continue
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "85bd815efd86af24338a0f0c45844cb7e1e6f155"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/85bd815efd86af24338a0f0c45844cb7e1e6f155/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Two Test Models

Analytical models such as the two-dimensional quadratic valley explain how curvature and gradient noise govern simple update rules, but evaluating modern optimizers requires empirical measurement on real neural networks. Why does AdamW outperform SGD by a wide margin on Transformers while offering a much smaller advantage on convolutional networks? Answering such questions requires training actual models under a controlled experimental protocol.

Because stochastic training runs vary across random initializations and minibatch draws, a difference between two optimizer configurations is meaningful only if it exceeds the run-to-run variation of a single configuration. This section defines the two benchmark architectures used throughout the rest of this chapter—the byte-level Transformer language model `TinyLM` ($470{,}784$ parameters) and the convolutional classifier `FashionCNN` ($421{,}642$ parameters)—along with our shared training loop, learning-rate sweep harness, baseline SGD measurements, and significance thresholds.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

In [ ]:
import os
import pandas as pd
import textwrap

## The Language Model TinyLM

`TinyLM` is an autoregressive byte-level language model that predicts the next UTF-8 byte ($V = 256$) given a context window of $64$ consecutive bytes (that section). A standard $2{,}000$-step run with minibatches of $64$ windows processes $2{,}000 \times 64 \times 64 \approx 8.2 \times 10^6$ target bytes. To avoid repeating training examples (which would conflate optimization speed with memorization), we train on the first shard of WikiText-103 [@Merity.Xiong.Bradbury.ea.2016] ($\sim 2.7 \times 10^8$ UTF-8 bytes of Wikipedia articles). Each training minibatch samples $64$ windows at random byte offsets, so a $2{,}000$-step run sees only $\sim 3\%$ of the corpus and every training step evaluates fresh text. Validation perplexity is measured on consecutive non-overlapping windows of the $1.1 \times 10^6$-byte validation split.

In [ ]:
class WikiText103(d2l.DataModule):
    """WikiText-103 as UTF-8 bytes, cut into windows of num_steps bytes."""
    def __init__(self, batch_size, num_steps, num_train=None, num_val=None):
        super().__init__()
        self.save_hyperparameters()
        self.vocab = d2l.BPETokenizer(vocab_size=256, specials=())  # Bytes
        self.train = self._read('train-00000-of-00002',
                                '3cbc792dfc0eeedf8d7333acdc15a9f536032d9a')
        self.val = self._read('validation-00000-of-00001',
                              '49d15019b0476afc752bd5f010b5a2aab2de89ca')
        self.train, self.val = self.train[:num_train], self.val[:num_val]

    def _read(self, split, sha1):
        url = ('https://huggingface.co/datasets/Salesforce/wikitext/resolve/'
               f'main/wikitext-103-raw-v1/{split}.parquet')
        fname = d2l.download(url, os.path.join(self.root, 'wikitext-103'),
                             sha1)
        lines = pd.read_parquet(fname)['text']  # Each line ends with '\n'
        return np.frombuffer(b''.join(s.encode() for s in lines), np.uint8)

    def windows(self, corpus, starts):
        """Inputs and next-byte targets of the windows at `starts`."""
        idx = np.asarray(starts)[:, None] + np.arange(self.num_steps + 1)
        array = corpus[idx].astype(np.int32)
        return array[:, :-1], array[:, 1:]

    def train_dataloader(self):  # A pass of minibatches at random offsets
        n = len(self.train) - self.num_steps
        rng = np.random.default_rng(d2l.get_seed())
        for _ in range(n // (self.batch_size * self.num_steps)):
            yield self.windows(self.train,
                               rng.integers(n, size=self.batch_size))

    def val_dataloader(self):  # Consecutive, non-overlapping windows
        starts = np.arange(0, len(self.val) - self.num_steps, self.num_steps)
        for i in range(0, len(starts), self.batch_size):
            yield self.windows(self.val, starts[i:i + self.batch_size])

In [ ]:
data = WikiText103(batch_size=64, num_steps=64)
print(f'{len(data.train):,} training bytes, '
      f'{len(data.val):,} validation bytes')
text = data.vocab.decode(data.train[:300])
print('\n'.join(textwrap.fill(line, 72) for line in text.splitlines()))

### Architecture and Parameter Groups

`TinyLM` is a two-block decoder-only Transformer (that section) with embedding width $d_{\text{model}} = 128$, $2$ causal attention heads, and a two-layer GELU MLP of hidden width $4 d_{\text{model}} = 512$ per block. Each sublayer is preceded by LayerNorm (that section) and wrapped in a residual connection, followed by a final LayerNorm and linear output head projecting to $256$ byte logits. Embedding rows are initialized from $\mathcal{N}(0, 1)$, and all linear weights and biases are drawn uniformly from $[-1/\sqrt{\text{fan-in}}, 1/\sqrt{\text{fan-in}}]$.

In [ ]:
class TinyLM(nnx.Module):
    """A small decoder-only transformer language model."""
    def __init__(self, vocab_size, d_model=128, num_heads=2, num_blks=2,
                 max_len=64, *, rngs):
        self.num_heads = num_heads
        def linear(d_in, d_out):
            a = 1 / math.sqrt(d_in)  # Weights and biases ~ U(-a, a)
            init = lambda key, shape, dtype: jax.random.uniform(
                key, shape, dtype, -a, a)
            return nnx.Linear(d_in, d_out, kernel_init=init, bias_init=init,
                              rngs=rngs)
        def embed(n):  # Embedding rows ~ N(0, I)
            return d2l.Embed(n, d_model, rngs=rngs,
                             embedding_init=nnx.initializers.normal(1.0))
        def norm():
            return nnx.LayerNorm(d_model, epsilon=1e-5, rngs=rngs)
        self.token_emb, self.pos_emb = embed(vocab_size), embed(max_len)
        self.blks = nnx.List([nnx.Dict(
            norm1=norm(), qkv=linear(d_model, 3 * d_model),
            proj=linear(d_model, d_model), norm2=norm(),
            mlp1=linear(d_model, 4 * d_model),
            mlp2=linear(4 * d_model, d_model))
            for _ in range(num_blks)])
        self.norm = norm()
        self.head = linear(d_model, vocab_size)

    def attention(self, blk, X):
        B, T, D = X.shape
        q, k, v = jnp.split(blk['qkv'](X), 3, axis=-1)
        q, k, v = (u.reshape(B, T, self.num_heads, -1) for u in (q, k, v))
        Y = jax.nn.dot_product_attention(q, k, v, is_causal=True)
        return blk['proj'](Y.reshape(B, T, D))

    def __call__(self, X):
        H = self.token_emb(X) + self.pos_emb(jnp.arange(X.shape[1]))
        for blk in self.blks:
            H = H + self.attention(blk, blk['norm1'](H))
            H = H + blk['mlp2'](jax.nn.gelu(blk['mlp1'](blk['norm2'](H)),
                                            approximate=False))
        return self.head(self.norm(H))

For an optimizer, the $470{,}784$ parameters of `TinyLM` divide into three distinct structural groups that later sections treat differently (e.g., for decoupled weight decay in that section and matrix orthogonalization in that section):

* **Embeddings ($9\%$):** lookup tables (`token_emb`, `pos_emb`) whose rows receive sparse gradients proportional to token frequency.
* **Weight matrices ($90\%$):** dense 2D projection matrices (`qkv`, `proj`, the two MLP layers per block, and `head`).
* **1D vectors ($<1\%$):** linear biases and LayerNorm scale/shift vectors.

In [ ]:
# Flax's default initializers differ, so TinyLM passes the ones described
# above (and approximate=False for the exact GELU); d2l.Embed computes
# the embedding gradient as one matrix product, far faster than XLA's
# default on frequent bytes such as the space
model = TinyLM(vocab_size=len(data.vocab), rngs=nnx.Rngs(d2l.get_key()))
kinds = {'embeddings': 0, 'matrices': 0, 'vectors': 0}
print(f'{"parameter":<22} {"shape":<14} {"count":>7}')
for path, p in nnx.to_flat_state(nnx.state(model, nnx.Param)):
    name = '.'.join(str(k) for k in path)
    kind = ('embeddings' if 'emb' in name else
            'matrices' if p.ndim == 2 else 'vectors')
    kinds[kind] += p.size
    print(f'{name:<22} {str(tuple(p.shape)):<14} {p.size:>7}')
for kind, count in kinds.items():
    print(f'{kind:>37} {count:>7}')
print(f'{"total":>37} {sum(kinds.values()):>7}')

## The Image Classifier FashionCNN

Because optimizer rankings on Transformers differ markedly from those on convolutional networks, our second benchmark is `FashionCNN`, a classifier for $28 \times 28$ Fashion-MNIST images (that section). Two $3 \times 3$ convolution + ReLU + $2 \times 2$ max-pooling stages ($32$ and $64$ channels) reduce the spatial resolution to $64 \times 7 \times 7$, followed by two linear layers ($3{,}136 \to 128 \to 10$). All layers use uniform $\pm 1/\sqrt{\text{fan-in}}$ initialization, totaling $421{,}642$ parameters.

In [ ]:
class FashionCNN(nnx.Module):
    """A compact CNN for Fashion-MNIST: two conv-pool stages, two linear."""
    def __init__(self, *, rngs):
        def init(fan_in):  # Weights and biases ~ U(-a, a), a = 1/sqrt(fan_in)
            a = 1 / math.sqrt(fan_in)
            f = lambda key, shape, dtype: jax.random.uniform(
                key, shape, dtype, -a, a)
            return dict(kernel_init=f, bias_init=f, rngs=rngs)
        self.conv1 = nnx.Conv(1, 32, kernel_size=(3, 3), **init(9))
        self.conv2 = nnx.Conv(32, 64, kernel_size=(3, 3), **init(9 * 32))
        self.fc1 = nnx.Linear(64 * 7 * 7, 128, **init(64 * 7 * 7))
        self.fc2 = nnx.Linear(128, 10, **init(128))

    def __call__(self, X):
        X = nnx.max_pool(nnx.relu(self.conv1(X)), window_shape=(2, 2),
                         strides=(2, 2))
        X = nnx.max_pool(nnx.relu(self.conv2(X)), window_shape=(2, 2),
                         strides=(2, 2))
        return self.fc2(nnx.relu(self.fc1(X.reshape(X.shape[0], -1))))

In [ ]:
# Flax's default initializers differ, so FashionCNN passes U(-a, a)
# explicitly; images arrive as (batch, height, width, channel)
print(f'{d2l.num_params(FashionCNN(rngs=nnx.Rngs(d2l.get_key()))):,} '
      'parameters')

With batch size $256$, a $2{,}000$-step run makes $\sim 8.5$ passes over the $60{,}000$ Fashion-MNIST training images, so we report held-out test accuracy alongside training loss via `evaluate_accuracy`:

In [ ]:
fashion = d2l.FashionMNIST(batch_size=256)

@nnx.jit
def num_correct(model, X, y):
    """Number of examples in (X, y) that model classifies correctly."""
    return (model(X).argmax(axis=1) == y).sum()

def evaluate_accuracy(model, batches):
    """Fraction of the examples in batches that model classifies correctly."""
    # A view with dropout off and batch norm on running statistics
    model = nnx.view(model, deterministic=True, use_running_average=True,
                     raise_if_not_found=False)
    correct, total = 0, 0
    for X, y in batches:
        correct += num_correct(model, jnp.asarray(X), jnp.asarray(y))
        total += len(y)
    return float(correct) / total

## Training Harness and Evaluation Helpers

Both models map a minibatch to logits and minimize mean cross-entropy loss over a fixed step budget. the table summarizes the training loop and evaluation utilities saved in `d2l`.

<!-- tbl-caption:Shared training and evaluation functions saved in `d2l`. -->

| Function | Purpose |
|:--|:--|
| `train_steps` | Train for `num_steps` steps (or until `stop(step)` holds) and return the per-step cross-entropy losses |
| `lm_perplexity` | Compute token perplexity $\exp(\text{mean cross-entropy})$ over an iterator of batches (that section) |
| `evaluate_accuracy` | Compute classification accuracy over an iterator of batches |
| `final_loss` | Mean of the last $k=100$ step losses (`inf` if diverged) |
| `smooth` | Block average over consecutive windows of $k=25$ losses for plotting |
| `steps_to_reach` | First step at which the $25$-step smoothed loss drops to `target` (or `None`) |
| `lr_sweep` | Train from identical initial weights across a list of learning rates and return loss curves and evaluation scores |

In [ ]:
@nnx.jit
def _cross_entropy_step(model, optimizer, X, Y):
    """One optimizer step on the mean cross-entropy of the logits model(X)
    for the labels Y; negative labels, such as -100, add no loss."""
    def loss_fn(model):
        logits = model(X)
        logits, y = logits.reshape(-1, logits.shape[-1]), Y.reshape(-1)
        valid = y >= 0
        losses = optax.softmax_cross_entropy_with_integer_labels(
            logits, jnp.where(valid, y, 0))
        return (losses * valid).sum() / valid.sum()
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)
    return loss

def train_steps(model, data, optimizer, num_steps, *, stop=None):
    """Minimize the mean cross-entropy of the logits model(X) for the labels
    Y over the minibatches (X, Y) of data, leaving out the labels -100, for
    num_steps steps or until stop(step) is true; return every step's loss."""
    losses, pending, step = [], [], 0
    while step < num_steps:
        for X, Y in data.train_dataloader():
            pending.append(_cross_entropy_step(
                model, optimizer, jnp.asarray(X), jnp.asarray(Y)))
            step += 1
            done = step >= num_steps or (stop is not None and stop(step))
            if len(pending) == 50 or done:  # Sync now and then
                losses.extend(float(l) for l in jax.device_get(pending))
                pending = []
            if done:
                return losses

In [ ]:
@jax.jit(static_argnums=0)
def lm_loss_sum(graphdef, state, X, Y):
    """Summed next-token cross-entropy of a batch, and the model's state."""
    model = nnx.merge(graphdef, state)
    loss = optax.softmax_cross_entropy_with_integer_labels(model(X), Y).sum()
    return loss, nnx.state(model)

def lm_perplexity(model, batches):
    """Perplexity of a language model on the minibatches (X, Y) of batches."""
    # A view with dropout off, sharing the weights; one walk through it
    graphdef, state = nnx.split(nnx.view(model, deterministic=True,
                                         raise_if_not_found=False))
    total, count = 0, 0
    for X, Y in batches:
        loss, state = lm_loss_sum(graphdef, state, X, Y)
        total, count = total + loss, count + Y.size
    nnx.update(model, state)  # Keep statistics the model records, if any
    return math.exp(float(total) / count)

In [ ]:
def final_loss(losses, k=100):
    """Mean of the last k losses; infinity if the run diverged."""
    v = sum(losses[-k:]) / k
    return v if math.isfinite(v) else float('inf')

def smooth(losses, k=25):
    """Means over consecutive blocks of k losses, for plotting."""
    return [sum(losses[i:i + k]) / k
            for i in range(0, len(losses) - k + 1, k)]

def steps_to_reach(losses, target, k=25):
    """First step at which the smoothed losses reach target; None if they
    never do."""
    for i, v in enumerate(smooth(losses, k)):
        if v <= target:
            return k * (i + 1)
    return None

In [ ]:
def lr_sweep(make_model, make_tx, data, lrs, seed,
             num_steps=2000, *, evaluate=None, metric='score', every=None,
             verbose=True):
    """Train make_model(rngs=...) with the optax transformation make_tx(lr)
    for each lr, from one initialization; return the loss curves and the
    evaluations."""
    every = every or num_steps
    curves, scores = {}, {}
    for lr in lrs:  # The same initialization for every lr
        model = make_model(rngs=nnx.Rngs(seed))
        optimizer = nnx.Optimizer(model, make_tx(lr), wrt=nnx.Param)
        curves[lr], scores[lr] = [], []
        for _ in range(num_steps // every):
            curves[lr] += train_steps(model, data, optimizer, every)
            if evaluate is not None:
                scores[lr].append(evaluate(model))
        line = f'lr {lr:g}: final loss {final_loss(curves[lr]):.3f}'
        if scores[lr]:
            line += f', {metric} {scores[lr][-1]:.3f}'
        if verbose:
            print(line)
    return curves, scores

## Baseline Minibatch SGD Runs and Experimental Protocol

We now establish our baseline with constant-rate minibatch SGD ($2{,}000$ steps, no momentum or weight decay). On `TinyLM`, an untrained model starts slightly above the uniform-distribution cross-entropy $\ln 256 \approx 5.55$ nats/byte (at $\sim 5.71$ nats/byte due to initial logit variance; Exercise 1). As reference landmarks on WikiText-103, a unigram byte-frequency table achieves $\sim 3.2$ nats/byte and a bigram byte-pair table achieves $\sim 2.5$ nats/byte (Exercise 6).

Sweeping plain SGD across $\eta \in \{0.1, 0.5, 2.5\}$ exhibits the three canonical learning-rate regimes:

In [ ]:
make_lm = lambda rngs: TinyLM(len(data.vocab), rngs=rngs)
sgd = optax.sgd
new_seed = d2l.get_seed

In [ ]:
lm_sgd, _ = lr_sweep(make_lm, sgd, data, lrs=[0.1, 0.5, 2.5], seed=new_seed())
d2l.plot(list(range(1, 2001)),
         list(lm_sgd.values()) + [[math.log(256)] * 2000],
         'step', 'training loss', xscale='log', ylim=[1.5, 6],
         legend=[f'lr {lr:g}' for lr in lm_sgd] + ['uniform guess'],
         legend_loc='outside', figsize=(5, 2.5))

* **Too high ($\eta = 2.5$):** Exceeds the local stability threshold $2/L$ after the first step and diverges to `inf` within ten steps.
* **Too low ($\eta = 0.1$):** Descends smoothly but slowly, reaching only $\sim 2.5$ nats/byte (bigram level) at step $2{,}000$.
* **Well-tuned ($\eta = 0.5$):** Experiences brief early transient spikes as the network navigates initial high-curvature regions, then recovers and reaches $\sim 2.13$ nats/byte.

On `FashionCNN`, plain SGD at $\eta = 0.3$ reaches a training loss of $\sim 0.22$ and a held-out test accuracy of $\sim 90\%$:

In [ ]:
test_acc = lambda model: evaluate_accuracy(model, fashion.val_dataloader())
cnn_sgd, _ = lr_sweep(FashionCNN, sgd, fashion, lrs=[0.3], seed=new_seed(),
                      evaluate=test_acc, metric='test accuracy')

### Comparison Protocol and Significance Thresholds

Following the Deep Learning Tuning Playbook [@Godbole.Dahl.Gilmer.ea.2023], comparing two optimizers fairly requires treating the learning rate $\eta$ as a **nuisance hyperparameter** that must be swept independently for each method (`lr_sweep` over a log-spaced grid from shared initial weights) rather than comparing two optimizers at an arbitrary shared learning rate [@Schmidt.Schneider.Hennig.2021]. Alongside the minimum loss achieved at each optimizer's best learning rate, we track its **usable learning-rate range** (grid points finishing within $0.15$ nats of the optimum).

Across $5$ independent random seeds at a well-tuned learning rate, the final $100$-step training loss (`final_loss`) and validation loss of `TinyLM` vary over a range of $\sim 0.01\text{--}0.015$ nats, while `FashionCNN` varies by $\sim 0.02$ in training loss and $\sim 1\%$ in test accuracy. Adopting a conservative $3\times$ range margin, we treat a single-run difference as statistically meaningful only if it exceeds **$0.05$ nats** on `TinyLM` or **$0.06$ training loss ($3\%$ test accuracy)** on `FashionCNN`; smaller differences are treated as ties unless confirmed across multiple seeds.

## Summary

* `TinyLM` ($470{,}784$ parameters: $90\%$ 2D projection matrices, $9\%$ embeddings, $<1\%$ 1D vectors) trains on random $64$-byte windows of WikiText-103 ($\sim 2.7 \times 10^8$ bytes), so a $2{,}000$-step run evaluates fresh text on every step.
* `FashionCNN` ($421{,}642$ parameters) trains on $256$-image minibatches of Fashion-MNIST ($\sim 8.5$ epochs in $2{,}000$ steps) and is evaluated by both training loss and held-out test accuracy.
* Every optimizer comparison sweeps the learning rate $\eta$ independently for each method from identical initial weights, reporting both the optimal loss and the usable learning-rate range against a multi-seed significance threshold of $0.05$ nats on `TinyLM`.

## Exercises

1. **Initial loss before training.**
    1. If logits $z_1, \ldots, z_V$ are drawn i.i.d. from $\mathcal{N}(0, s^2)$ independently of the target, use $\sum_{i=1}^V e^{z_i} \approx V \mathbb{E}[e^{z_i}] = V e^{s^2/2}$ to show that the expected initial cross-entropy is approximately $\ln V + s^2 / 2$.
    1. Given LayerNorm outputs of mean $0$ and variance $1$ and head weights/biases drawn from $\mathcal{U}(-1/\sqrt{128}, 1/\sqrt{128})$, show that $s^2 = 129/384 \approx 1/3$ and $\ln 256 + 1/6 \approx 5.71$ nats, matching the step-$1$ loss of `TinyLM`. What is the initial loss if the head weights and bias are initialized to zero?

1. [code] **Diagnosing loss curves on `FashionCNN`.** Train `FashionCNN` with plain SGD across $\eta \in \{0.003, 0.03, 0.3, 1.0\}$ and plot the step losses on a logarithmic step axis against the uniform baseline $\ln 10 \approx 2.3026$. For the run that stalls near $\ln 10$, inspect the fraction of dead ReLU activations across a minibatch to explain why gradients vanish.

1. [code] **The edge of divergence on `TinyLM`.** Sweep plain SGD on `TinyLM` across $\eta \in \{0.7, 1.0, 1.4, 2.0\}$ across $5$ seeds (`lr_sweep(..., verbose=False)`). Record the survival rate, the step at which divergent runs fail, and the lowest surviving loss.

1. [code] **Multi-seed spread and standard error.** ● Run $20$ seeds of `TinyLM` with `torch.optim.AdamW` / `optax.adamw` at $\eta = 3 \times 10^{-3}$ (`weight_decay=0`), compare the sample standard deviation $\sigma$ of `final_loss` against the normal range estimator $\text{range}/3.7$, and compute how many seeds $k$ are required to detect a $0.005$-nat difference at two standard errors ($\sigma\sqrt{2/k}$).

1. [code] **Training loss vs. validation loss and overfitting small corpora.**
    1. Explain why `final_loss` (averaged over the last $100$ training steps) is slightly higher ($0.02\text{--}0.03$ nats) than the post-step-$2000$ validation loss on `TinyLM`, and verify by evaluating the final checkpoint on held-out training windows.
    1. Restrict the corpus to $100{,}000$ bytes via `WikiText103(batch_size=64, num_steps=64, num_train=100000)` ($\sim 80$ epochs in $2{,}000$ steps) and plot training vs. validation loss every $250$ steps (`every=250`).

1. [code] **Count-based $n$-gram landmarks.** Compute the empirical byte unigram entropy ($\sim 3.2$ nats) and add-one smoothed bigram ($\sim 2.5$ nats) and trigram cross-entropies from the first $10^7$ training bytes of `WikiText103` evaluated on the validation split, and compare them against the $2{,}000$-step SGD ($2.13$ nats) and AdamW ($1.59$ nats) losses of `TinyLM`.

[Discussions](https://d2l.discourse.group/t/36884)